# Mondkrater Mini-Test (NASA-IBM Lunar Foundation Model)

Ziel: Bevor ein vollständiger Trainingslauf startet, messen wir, **wie viel Grafikspeicher (VRAM)** und **wie viel Zeit pro Durchgang (Epoche)** das Training braucht.

* **Stufe A, Erkennen:** NASAs fertiges Krater-Modell läuft über 4 Testkacheln. Nur ein Ladetest: Die Zahl soll grob in der Gegend von NASAs 0,258 liegen, mit 4 Kacheln schwankt sie stark.
* **Stufe B, kurz selbst trainieren:** 2 Epochen auf 12 Kacheln mit der offiziellen Konfiguration, in 5 Speicher-Stufen. Die Genauigkeit danach ist bedeutungslos, es geht nur um VRAM und Zeit.

Technik: Das NASA-Repo braucht Python 3.11 oder 3.12, Colab läuft auf 3.13. Zelle 2 baut deshalb eine eigene Python-3.12-Umgebung (`/content/py312`) mit GPU-PyTorch. Alle Rechenschritte laufen darin, ein Neustart ist nicht nötig.

Daten: Modell und Grundmodell von Hugging Face (Apache-2.0). Die 20 Kacheln kommen aus `Sombench-WAC-Crater-Detection` (CC BY 4.0) und werden **nur zur Laufzeit** geladen. Bilder daraus nicht speichern oder teilen.

**So geht's:** *Laufzeit → Laufzeittyp ändern → T4 GPU → Speichern*, dann *Laufzeit → Alle ausführen*. Dauer ca. 25 bis 40 Minuten. Am Ende die Ausgabe der letzten Zelle prüfen.

In [ ]:
# 0) Grafik-Backend festlegen: Colab setzt eines, das nur im Notebook-Kernel existiert und Hilfsprozesse abstürzen ließe
import os
os.environ["MPLBACKEND"] = "Agg"
print("MPLBACKEND =", os.environ["MPLBACKEND"])

### 1) Skripte schreiben

In [ ]:
%%writefile setup_env.sh
#!/usr/bin/env bash
# Separate Python 3.12 environment for the NASA repo (it requires Python >=3.11,<3.13;
# Colab's kernel is 3.13). The notebook kernel itself is not touched, so no restart is needed.
# Usage: bash setup_env.sh [--force-torch]   (--force-torch: reinstall the CUDA torch wheels)
# Env overrides (for testing elsewhere): VENV, TORCH_INDEX, REQUIRE_CUDA=0 (accept a CPU torch build)
set -euo pipefail
export MPLBACKEND=Agg  # Colab exports an inline backend that crashes matplotlib outside the kernel
VENV="${VENV:-/content/py312}"
# cu126 wheels: contain kernels for the T4 (sm_75) and run on every driver >= 525, incl. newer 5xx drivers.
TORCH_INDEX="${TORCH_INDEX:-https://download.pytorch.org/whl/cu126}"
REQUIRE_CUDA="${REQUIRE_CUDA:-1}"
UV_VERSION="${UV_VERSION:-0.13.8}"
REPO_REF="${REPO_REF:-d54c67aad513cb9daca444afa425cfb278e4fbf8}"  # pinned upstream commit (20.09.2026)
REPO="git+https://github.com/NASA-IMPACT/NASA-IBM-Lunar-Foundation-Model.git@$REPO_REF"
PY="$VENV/bin/python"

status() {  # always printed, also on the "already done" path
  "$PY" - <<'PY'
import sys, torch, importlib.metadata as m
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA-Build", torch.version.cuda,
      "| terratorch", m.version("terratorch"), "| GPU sichtbar:", torch.cuda.is_available())
PY
}
repo_ok()  { [ -x "$PY" ] && "$PY" -c "import ni_lfm, terratorch_integration" 2>/dev/null; }
torch_ok() {  # importable torch; with REQUIRE_CUDA=1 it must be a CUDA build (CPU-only wheels have torch.version.cuda None)
  [ -x "$PY" ] && "$PY" -c "import torch, torchvision, sys; sys.exit(0 if ($REQUIRE_CUDA == 0 or torch.version.cuda) else 1)" 2>/dev/null; }
install_torch() {
  echo "Installiere PyTorch-Räder von $TORCH_INDEX ..."
  uv pip install --quiet --python "$PY" --reinstall-package torch --reinstall-package torchvision \
    --index-url "$TORCH_INDEX" "torch==2.12.1" "torchvision==0.27.1"
  # Pin these exact builds so the repo install cannot swap them for other CUDA builds.
  "$PY" -c "import torch, torchvision; print(f'torch=={torch.__version__}\ntorchvision=={torchvision.__version__}')" > "$VENV/pins.txt"
}
install_repo() {
  uv pip install --quiet --python "$PY" --constraint "$VENV/pins.txt" \
    --index-url https://pypi.org/simple --extra-index-url "$TORCH_INDEX" --index-strategy unsafe-best-match \
    "$REPO" "terratorch[peft]==1.2.13"
}

FORCE=0; [ "${1:-}" = "--force-torch" ] && FORCE=1
if [ $FORCE = 0 ] && repo_ok && torch_ok; then
  echo "Umgebung $VENV ist schon fertig."; status; exit 0; fi
pip -q install "uv==${UV_VERSION}"
[ -x "$PY" ] || uv venv --quiet --python 3.12 "$VENV"
if [ $FORCE = 1 ] || ! torch_ok; then install_torch; fi
[ -f "$VENV/pins.txt" ] || install_torch
repo_ok || install_repo
torch_ok || { echo "FEHLER: torch in $VENV ist kein CUDA-Build."; status; exit 1; }
status


In [ ]:
%%writefile download.py
"""Downloads at runtime: crater model (Apache-2.0, 1.1 GB), backbone (Apache-2.0, 2.4 GB). Usage: python download.py HF_DIR"""
import os
os.environ["MPLBACKEND"] = "Agg"  # Colab sets an inline backend that only exists in its kernel
import sys
from huggingface_hub import hf_hub_download, snapshot_download
d = sys.argv[1]; R = "nasa-ibm-ai4science/Crater-Detection-NASA-IBM-Lunar-Foundation-Model"
for f in ["WAC_ni_lfm_ps8_lora_s46.ckpt", "WAC_config.yaml"]:
    print(hf_hub_download(R, f, local_dir=d))
snapshot_download("nasa-ibm-ai4science/NASA-IBM-Lunar-Foundation-Model", allow_patterns="backbone/*", local_dir=d)
print("fertig")


In [ ]:
%%writefile make_mini_split.py
"""Download a 20-tile mini split of Sombench-WAC-Crater-Detection (CC BY 4.0) at runtime.
Nothing from this dataset is stored in our repo. Default: 12 train / 4 val / 4 test, fixed seed.
Usage: python make_mini_split.py OUT_DIR [n_train n_val n_test]"""
import os
os.environ["MPLBACKEND"] = "Agg"  # Colab sets an inline backend that only exists in its kernel
import sys, json, random, os
from huggingface_hub import hf_hub_download
REPO = "nasa-ibm-ai4science/Sombench-WAC-Crater-Detection"
out = sys.argv[1]; n = dict(zip(["train", "val", "test"], map(int, sys.argv[2:5] or [12, 4, 4])))
os.makedirs(out, exist_ok=True)
hf_hub_download(REPO, "metadata.parquet", repo_type="dataset", local_dir=out)
for split, k in n.items():
    full = json.load(open(hf_hub_download(REPO, f"{split}.json", repo_type="dataset", local_dir=out)))
    per = {}
    for a in full["annotations"]:
        per[a["image_id"]] = per.get(a["image_id"], 0) + 1
    imgs = sorted([i for i in full["images"] if per.get(i["id"], 0) >= 20], key=lambda i: i["id"])
    pick = random.Random(46).sample(imgs, k); ids = {i["id"] for i in pick}
    mini = {kk: v for kk, v in full.items() if kk not in ("images", "annotations")}
    mini["images"] = pick; mini["annotations"] = [a for a in full["annotations"] if a["image_id"] in ids]
    json.dump(mini, open(f"{out}/mini_{split}.json", "w"))
    for i in pick:
        stem = os.path.splitext(os.path.basename(i["file_name"]))[0]
        hf_hub_download(REPO, f"images_tiff/{stem}.tif", repo_type="dataset", local_dir=out)
    print(split, len(pick), "tiles,", len(mini["annotations"]), "annotations")


In [ ]:
%%writefile stage_a.py
"""Stage A: NASA's released WAC crater checkpoint, inference only, on N test tiles.
Uses the upstream data module + test_step, so the metric path is identical to the paper's.
Usage: python stage_a.py CKPT BACKBONE_DIR DATA_DIR ANN_JSON [--plot out.png]
The optional plot writes one tile with reference craters and model boxes."""
import os
os.environ["MPLBACKEND"] = "Agg"  # Colab sets an inline backend that only exists in its kernel
import sys, time, copy, os, json, argparse
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import torch, lightning.pytorch as pl
import terratorch_integration  # registers backbone/necks/task
from terratorch_integration import LunarObjectDetectionTask
from terratorch_integration.data_adapter import LunarWACCraterDataModule

p = argparse.ArgumentParser(); p.add_argument("ckpt"); p.add_argument("backbone_dir"); p.add_argument("data_dir"); p.add_argument("ann"); p.add_argument("--plot")
a = p.parse_args()
CKPT = a.ckpt; CFG = f"{a.backbone_dir}/config.yaml"
MEANS = [0.018106, 0.025262, 0.027113, 0.028616, 0.030757]   # from NASA WAC_config.yaml (train split)
STDS = [0.010051, 0.013646, 0.014491, 0.015267, 0.016555]

# Lightning stores the checkpoint as a pickle, so weights_only=True cannot read it.
# Only the pinned NASA crater checkpoint, or a patched copy this script wrote, is loaded.
ck = torch.load(CKPT, map_location="cpu", weights_only=False)
model_args = copy.deepcopy(ck["hyper_parameters"]["model_args"])
# The released ckpt stores one kwarg ('use_resolution_conditioning') that public terratorch 1.2.x
# does not accept. Strip it once into a patched copy; weights are untouched.
hp = ck["hyper_parameters"]
if "use_resolution_conditioning" in hp or "use_resolution_conditioning" in hp.get("_instantiator_kwargs", {}) or \
   "use_resolution_conditioning" in str(hp):
    def strip(d):
        if isinstance(d, dict):
            d.pop("use_resolution_conditioning", None)
            for v in d.values(): strip(v)
        elif isinstance(d, list):
            for v in d: strip(v)
        elif hasattr(d, "__dict__"):
            for k in list(vars(d)):
                if k == "use_resolution_conditioning": delattr(d, k)
                else: strip(getattr(d, k))
    strip(hp)
    PATCHED = CKPT.replace(".ckpt", "_patched.ckpt")
    if not os.path.exists(PATCHED): torch.save(ck, PATCHED)
    CKPT = PATCHED
del ck
model_args["backbone_cfg"] = CFG; model_args["backbone_checkpoint_path"] = None  # ckpt carries weights
model_args.pop("backbone_modality_info_path", None)  # absolute path on NASA cluster; not needed
t0 = time.time()
model = LunarObjectDetectionTask.load_from_checkpoint(CKPT, map_location="cpu", model_args=model_args).eval()
print(f"model loaded in {time.time()-t0:.1f}s")

dm = LunarWACCraterDataModule(data_dir=a.data_dir, train_annotations_file=a.ann, val_annotations_file=a.ann,
                              test_annotations_file=a.ann, batch_size=1, num_workers=0, image_size=512,
                              min_diameter_px=10.0, drop_no_crater_images=True,
                              wac_norm_means=MEANS, wac_norm_stds=STDS)
acc = "gpu" if (torch.cuda.is_available() or torch.backends.mps.is_available()) else "cpu"
trainer = pl.Trainer(accelerator=acc, devices=1, logger=False, enable_checkpointing=False, precision="32")
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else acc)
t0 = time.time(); res = trainer.test(model, datamodule=dm, verbose=False); dt = time.time() - t0
n = len(dm.test_dataset) if hasattr(dm, "test_dataset") else None
out = {"accelerator": acc, "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else acc,
       "gpu_total_GB": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else None, "tiles": n, "seconds_total": round(dt, 1),
       "seconds_per_tile": round(dt / n, 1) if n else None,
       "test_mAP": float(res[0].get("test_map", float("nan"))),
       "test_mAP_50": float(res[0].get("test_map_50", float("nan"))),
       "ref_craters": int(sum(len(dm.test_dataset[i]["boxes"]) for i in range(n))) if n else None,
       "peak_vram_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None}
print("RAW", {k: float(v) for k, v in res[0].items()})
print(json.dumps(out, indent=1))
json.dump(out, open(os.environ.get("STAGE_A_OUT", "stage_a_result.json"), "w"), indent=1)

if a.plot:  # one tile: reference craters (grey) vs model boxes >= 0.5 (yellow)
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt, matplotlib.patches as P
    s = dm.test_dataset[0]; x = s["vis"] if "vis" in s else s["image"]
    dev = next(model.parameters()).device
    with torch.no_grad():
        y = model.model(x[None].to(dev)); y = y.output[0] if hasattr(y, "output") else y[0]
    img = x[2].numpy(); fig, ax = plt.subplots(figsize=(7, 7)); ax.imshow(img, cmap="gray"); ax.axis("off")
    for b in s["boxes"].tolist():
        ax.add_patch(P.Circle(((b[0]+b[2])/2, (b[1]+b[3])/2), (b[2]-b[0])/2, fill=False, ec="#9a9a9a", lw=0.8))
    k = 0
    for b, sc in zip(y["boxes"].cpu().tolist(), y["scores"].cpu().tolist()):
        if sc >= 0.5:
            k += 1; ax.add_patch(P.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], fill=False, ec="#F5C400", lw=1.1))
    ax.set_title(f"Referenz (grau): {len(s['boxes'])}  ·  Modell >=0.5 (gelb): {k}")
    fig.savefig(a.plot, dpi=110, bbox_inches="tight"); print("plot", a.plot, "ref", len(s["boxes"]), "pred", k)


In [ ]:
%%writefile stage_b.py
"""Stage B: short LoRA fine-tune from NASA's pretrained backbone, official WAC config values.
Measures seconds per epoch and peak GPU memory. Mini-test only: 2 epochs on 12 tiles says
nothing about final accuracy; it tells us whether and how fast the real run fits the GPU.
Usage: python stage_b.py CONFIG_YAML HF_BACKBONE_DIR MINI_DIR [--epochs 2 --batch 4 --accum 1
       --precision 32|amp|bf16-mixed|16-mixed (amp = bf16 if the GPU supports it, else fp16) --patch 8 --deterministic 1 --limit-batches 0]"""
import os
os.environ["MPLBACKEND"] = "Agg"  # Colab sets an inline backend that only exists in its kernel
import os, sys, time, json, argparse, yaml, torch, lightning.pytorch as pl
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import terratorch_integration
from terratorch_integration import LunarObjectDetectionTask, DeterministicAugmentation
from terratorch_integration.data_adapter import LunarWACCraterDataModule
from terratorch_integration.data_utils import D4DetectionTransform

p = argparse.ArgumentParser()
p.add_argument("config"); p.add_argument("backbone_dir"); p.add_argument("mini_dir")
p.add_argument("--epochs", type=int, default=2); p.add_argument("--batch", type=int, default=4)
p.add_argument("--accum", type=int, default=1); p.add_argument("--precision", default="32")
p.add_argument("--patch", type=int, default=8); p.add_argument("--deterministic", type=int, default=1)
p.add_argument("--limit-batches", type=int, default=0); p.add_argument("--out", default="stage_b_result.json")
a = p.parse_args()

cfg = yaml.safe_load(open(a.config)); pl.seed_everything(cfg.get("seed_everything", 46))
m = cfg["model"]["init_args"]; d = cfg["data"]["init_args"]
m["model_args"]["backbone_checkpoint_path"] = f"{a.backbone_dir}/checkpoint.pt"
m["model_args"]["backbone_cfg"] = f"{a.backbone_dir}/config.yaml"
if a.patch != 8:  # patch 16 = 4x fewer tokens, the low-VRAM fallback; ReshapeTokensToImage h must match
    m["model_args"]["backbone_patch_size"] = a.patch
    for n in m["model_args"]["necks"]:
        if n["name"] == "ReshapeTokensToImage": n["h"] = d["image_size"] // a.patch
model = LunarObjectDetectionTask(**m)

dm = LunarWACCraterDataModule(
    data_dir=a.mini_dir, train_annotations_file=f"{a.mini_dir}/mini_train.json",
    val_annotations_file=f"{a.mini_dir}/mini_val.json", test_annotations_file=f"{a.mini_dir}/mini_test.json",
    batch_size=a.batch, num_workers=2, image_size=d["image_size"], min_diameter_px=d["min_diameter_px"],
    drop_no_crater_images=d["drop_no_crater_images"], wac_norm_means=d["wac_norm_means"],
    wac_norm_stds=d["wac_norm_stds"], train_transforms=D4DetectionTransform(image_keys=["image", "vis"], p=1.0))

class Clock(pl.Callback):
    def __init__(s): s.epochs = []
    def on_train_epoch_start(s, *_):
        if torch.cuda.is_available(): torch.cuda.synchronize()
        s.t = time.time()
    def on_train_epoch_end(s, *_):
        if torch.cuda.is_available(): torch.cuda.synchronize()
        s.epochs.append(round(time.time() - s.t, 1))

clock = Clock()
acc = "gpu" if (torch.cuda.is_available() or torch.backends.mps.is_available()) else "cpu"
lim = a.limit_batches or None
if a.precision == "amp":  # T4 (Turing) has no bf16 -> fp16 mixed
    # is_bf16_supported() can be True on a T4 (sm_75) via emulation, which is slow; native bf16 needs Ampere (sm_80)+.
    a.precision = "bf16-mixed" if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else "16-mixed"
    if acc == "cpu": a.precision = "bf16-mixed"
trainer = pl.Trainer(accelerator=acc, devices=1, precision=a.precision, max_epochs=a.epochs,
                     accumulate_grad_batches=a.accum, deterministic=bool(a.deterministic), benchmark=False,
                     logger=False, enable_checkpointing=False, log_every_n_steps=1, num_sanity_val_steps=0,
                     callbacks=[DeterministicAugmentation(), clock],
                     limit_train_batches=lim, limit_val_batches=lim, limit_test_batches=lim)
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
t0 = time.time(); trainer.fit(model, datamodule=dm); fit_s = time.time() - t0
res = trainer.test(model, datamodule=dm, verbose=False)[0]
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else acc
out = {"device": gpu,
       "gpu_total_GB": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else None,
       "peak_vram_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None,
       "peak_vram_reserved_GB": round(torch.cuda.max_memory_reserved() / 1e9, 2) if torch.cuda.is_available() else None,
       "precision": a.precision, "deterministic": bool(a.deterministic),
       "compute_capability": ".".join(map(str, torch.cuda.get_device_capability(0))) if torch.cuda.is_available() else None, "batch": a.batch, "accum": a.accum, "patch": a.patch,
       "train_tiles": len(dm.train_dataset), "seconds_per_epoch": clock.epochs, "fit_seconds_total": round(fit_s, 1),
       "test_mAP_after_mini_finetune": float(res.get("test_map", float("nan")))}
print(json.dumps(out, indent=1)); json.dump(out, open(a.out, "w"), indent=1)


In [ ]:
%%writefile run_ladder.py
"""Stage B memory ladder: each rung runs in its own process, so an out-of-memory error only stops that rung.
Usage: python run_ladder.py CONFIG BACKBONE_DIR MINI_DIR [--epochs 2] [--limit-batches 0] [--only name,...]"""
import os
os.environ["MPLBACKEND"] = "Agg"  # Colab sets an inline backend that only exists in its kernel
import subprocess, json, sys, argparse
p = argparse.ArgumentParser(); p.add_argument("config"); p.add_argument("backbone"); p.add_argument("mini")
p.add_argument("--epochs", default="2"); p.add_argument("--limit-batches", default="0"); p.add_argument("--only", default="")
a = p.parse_args()
LADDER = [  # name, precision, batch, accumulate, patch, deterministic
    ("offiziell_fp32_b4", "32", 4, 1, 8, 1),
    ("amp_b4", "amp", 4, 1, 8, 1),
    ("amp_b2_acc2", "amp", 2, 2, 8, 1),
    ("amp_b1_acc4", "amp", 1, 4, 8, 1),
    # deterministic=1 makes torchvision use a pure-PyTorch roi_align on CUDA, which needs much more memory.
    ("amp_b1_acc4_nondet", "amp", 1, 4, 8, 0),
    ("amp_ps16_b2_acc2", "amp", 2, 2, 16, 1),
    ("amp_ps16_b4_nondet", "amp", 4, 1, 16, 0),
]
results = {}
for name, prec, b, acc, ps, det in LADDER:
    if a.only and name not in a.only.split(","): continue
    cmd = [sys.executable, "stage_b.py", a.config, a.backbone, a.mini, "--epochs", a.epochs, "--batch", str(b),
           "--accum", str(acc), "--precision", prec, "--patch", str(ps), "--deterministic", str(det), "--limit-batches", a.limit_batches,
           "--out", f"b_{name}.json"]
    r = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ, "MPLBACKEND": "Agg"})
    if r.returncode == 0:
        results[name] = json.load(open(f"b_{name}.json"))
        print(name, "OK |", results[name]["precision"], "|", results[name]["peak_vram_GB"], "GB |", results[name]["seconds_per_epoch"], "s/Epoche", flush=True)
    else:
        err = r.stderr + r.stdout
        if "OutOfMemory" in err or "out of memory" in err:
            msg = "OOM (GPU-Speicher voll)"
        elif r.returncode in (-9, 137):
            msg = "Prozess beendet, vermutlich Arbeitsspeicher (RAM) voll"
        else:
            lines = [l for l in err.splitlines() if "Error" in l or "error:" in l] or err.strip().splitlines() or ["?"]
            msg = lines[-1][:300]
        results[name] = {"error": msg, "returncode": r.returncode}
        print(name, "FEHLER", results[name]["error"], flush=True)
json.dump(results, open("stage_b_ladder.json", "w"), indent=1)


In [ ]:
# 2) GPU prüfen und Python-3.12-Umgebung bauen (einmalig ca. 3-5 min, beim zweiten Mal sofort fertig)
import subprocess, sys, os
PY = "/content/py312/bin/python"
ENV = {**os.environ, "MPLBACKEND": "Agg"}
def sh(cmd):
    return subprocess.run(cmd, capture_output=True, text=True, env=ENV)
print("Kernel-Python:", sys.version.split()[0])
gpu = sh(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"])
assert gpu.returncode == 0, "Keine GPU gefunden: Laufzeit -> Laufzeittyp ändern -> T4 GPU, dann erneut 'Alle ausführen'."
print(gpu.stdout)

def setup(*args):
    r = sh(["bash", "setup_env.sh", *args]); print(r.stdout[-2000:])
    assert r.returncode == 0, r.stderr[-3000:]

CHECK = "import torch; print('GPU sichtbar:', torch.cuda.is_available(), '| torch', torch.__version__, '| CUDA-Build', torch.version.cuda)"
def gpu_ok():  # eigener Test, unabhängig von der Ausgabe von setup_env.sh
    r = sh([PY, "-c", CHECK]); print((r.stdout + r.stderr).strip()[-1500:])
    return r.returncode == 0 and "GPU sichtbar: True" in r.stdout

setup()
if not gpu_ok():
    print("PyTorch sieht die GPU nicht -> installiere die CUDA-Räder (cu126) neu und prüfe erneut ...")
    setup("--force-torch")
    if not gpu_ok():
        print("--- Diagnose ---")
        for cmd in (["nvidia-smi"], [PY, "-c", "import torch; torch.cuda.init()"],
                    ["bash", "-c", "echo CUDA_VISIBLE_DEVICES=$CUDA_VISIBLE_DEVICES; echo LD_LIBRARY_PATH=$LD_LIBRARY_PATH; uv pip list --python " + PY + " 2>/dev/null | grep -iE 'torch|nvidia' || true"]):
            r = sh(cmd); print("$", " ".join(cmd)[:120]); print((r.stdout + r.stderr)[-2500:])
        raise AssertionError("PyTorch sieht die GPU auch nach Neuinstallation nicht. Bitte die komplette Ausgabe dieser Zelle zurückschicken.")
print("GPU bereit.")

In [ ]:
# 3) Downloads: Krater-Modell (1,1 GB), Grundmodell (2,4 GB), 20 Kacheln (~80 MB) + Annotationen
!MPLBACKEND=Agg {PY} download.py hf
!MPLBACKEND=Agg {PY} make_mini_split.py wac_mini 12 4 4

In [ ]:
# 4) Stufe A: NASAs Krater-Modell auf 4 Testkacheln
!MPLBACKEND=Agg STAGE_A_OUT=stage_a_result.json {PY} stage_a.py hf/WAC_ni_lfm_ps8_lora_s46.ckpt hf/backbone wac_mini wac_mini/mini_test.json 2>&1 | grep -vi warn | tail -16

In [ ]:
# 5) Stufe B: Speicher-Leiter (je 2 Epochen). Ein Speicherüberlauf (OOM) bricht nur die jeweilige Stufe ab.
LIMIT = 0  # 0 = alle Kacheln
!MPLBACKEND=Agg {PY} run_ladder.py hf/WAC_config.yaml hf/backbone wac_mini --epochs 2 --limit-batches {LIMIT}

In [ ]:
# 6) Zusammenfassung zum Zurückschicken (bitte diese Ausgabe komplett kopieren)
import json
a = json.load(open("stage_a_result.json")); b = json.load(open("stage_b_ladder.json"))
for v in b.values():
    if "seconds_per_epoch" in v:
        s = v["seconds_per_epoch"][-1] / v["train_tiles"]   # letzte Epoche, ohne Aufwärmen
        v["hochrechnung_stunden_800_kacheln_100_epochen"] = round(s * 800 * 100 / 3600, 1)  # grob, ohne Validierung
print(json.dumps({"gpu": a.get("device"), "stage_A": a, "stage_B": b}, indent=1))